### load datasets 

In [45]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

In [46]:
registry = pd.read_csv("../data/transformer_registry.csv")
oil = pd.read_csv("../data/oil_thermal_readings.csv")
load = pd.read_csv("../data/load_logs.csv")
labels = pd.read_csv("../data/fault_status_labeled.csv")
prediction = pd.read_csv("../data/fault_status_to_predict.csv")

In [47]:
# making copies 

registry_clean = registry.copy()
oil_clean = oil.copy()
load_clean = load.copy()
labels_clean = labels.copy()
prediction_clean = prediction.copy()

In [48]:
# Convert date columns

oil_clean["reading_date"] = pd.to_datetime(
    oil_clean["reading_date"]
)

load_clean["log_date"] = pd.to_datetime(
    load_clean["log_date"]
)

In [49]:
print(oil_clean["reading_date"].dtype)
print(load_clean["log_date"].dtype)

datetime64[ns]
datetime64[ns]


In [50]:
# Handle the suspicious dissolved gas values - negative values 

negative_gas = oil_clean[
    oil_clean["dissolved_gas_ppm"] < 0
]

negative_gas

,record_id,transformer_id,reading_date,dissolved_gas_ppm,oil_temp_c
396,OIL-000983,TXR-0143,2023-06-22,-104.634461,39.956458
488,OIL-000562,TXR-0083,2023-08-31,-130.905809,56.915574
526,OIL-000297,TXR-0047,2024-03-14,-138.477238,54.548677
659,OIL-000598,TXR-0088,2023-01-22,-130.033005,41.448063
857,OIL-000960,TXR-0138,2024-03-26,-113.603417,55.112288
968,OIL-000618,TXR-0091,2024-02-06,-64.327410,60.014134
1016,OIL-000485,TXR-0072,2023-07-10,-110.339867,46.174378
1064,OIL-000895,TXR-0130,2023-10-06,-103.347756,47.831253
1247,OIL-000022,TXR-0004,2023-12-13,-146.538586,37.887465
1287,OIL-000714,TXR-0105,2023-05-23,-77.567629,59.110396


In [51]:
print("Number of negative values:",
      len(negative_gas))

Number of negative values: 11


In [52]:
display(negative_gas["dissolved_gas_ppm"].describe())

count     11.000000
mean    -109.644524
std       26.015361
min     -146.538586
25%     -130.469407
50%     -110.339867
75%      -94.831171
max      -64.327410
Name: dissolved_gas_ppm, dtype: float64

In [53]:
#Replace negative gas values with NaN

oil_clean.loc[
    oil_clean["dissolved_gas_ppm"] < 0,
    "dissolved_gas_ppm"
] = np.nan

In [54]:
print(
    "Missing dissolved gas values:",
    oil_clean["dissolved_gas_ppm"].isna().sum()
)

Missing dissolved gas values: 11


In [55]:
print(
    "Negative dissolved gas values:",
    (oil_clean["dissolved_gas_ppm"] < 0).sum()
)

Negative dissolved gas values: 0


In [56]:
# Handle missing cooling_type
missing_cooling = registry_clean[
    registry_clean["cooling_type"].isna()
]

missing_cooling


,transformer_id,install_year,rated_kva,cooling_type,zone
9,TXR-0010,2014,100,NaN,Zone 3
21,TXR-0022,2009,500,NaN,Zone 2
41,TXR-0042,2022,100,NaN,Zone 2
69,TXR-0070,2009,1000,NaN,Zone 2
107,TXR-0108,2016,500,NaN,Zone 2
137,TXR-0138,2000,500,NaN,Zone 3
157,TXR-0158,2015,1000,NaN,Zone 2
188,TXR-0189,2007,250,NaN,Zone 3


In [57]:
print("Missing cooling type:",
      registry_clean["cooling_type"].isna().sum())

Missing cooling type: 8


In [58]:
# impute missing values as "Unknown"

registry_clean["cooling_type"] = registry_clean["cooling_type"].fillna("Unknown")

In [59]:
registry_clean["cooling_type"].value_counts()

ONAF       76
OFAF       75
ONAN       73
Unknown     8
Name: cooling_type, dtype: int64

In [60]:
# Missing load values

missing_load = load_clean[
    load_clean["load_pct_of_rated"].isna()
]

missing_load

,record_id,transformer_id,log_date,load_pct_of_rated
36,LOA-000706,TXR-0082,2024-03-04,NaN
42,LOA-001175,TXR-0137,2023-10-08,NaN
59,LOA-000320,TXR-0037,2024-04-03,NaN
129,LOA-000401,TXR-0046,2024-01-19,NaN
133,LOA-000630,TXR-0073,2023-10-11,NaN
156,LOA-001744,TXR-0206,2023-04-10,NaN
180,LOA-001884,TXR-0222,2023-03-06,NaN
229,LOA-000139,TXR-0017,2023-05-03,NaN
257,LOA-001584,TXR-0186,2023-05-25,NaN
260,LOA-001124,TXR-0131,2023-07-06,NaN


In [61]:
print(
    "Missing load values:",
    load_clean["load_pct_of_rated"].isna().sum()
)

Missing load values: 34


## Aggregation 

In [62]:
# Aggregate oil_thermal_readings

print("Oil missing values:")
print(oil_clean.isna().sum())

print("\nLoad missing values:")
print(load_clean.isna().sum())

Oil missing values:
record_id             0
transformer_id        0
reading_date          0
dissolved_gas_ppm    11
oil_temp_c            0
dtype: int64

Load missing values:
record_id             0
transformer_id        0
log_date              0
load_pct_of_rated    34
dtype: int64


In [63]:
#basic aggregation

oil_agg = oil_clean.groupby("transformer_id").agg(
    gas_mean=("dissolved_gas_ppm", "mean"),
    gas_max=("dissolved_gas_ppm", "max"),
    gas_std=("dissolved_gas_ppm", "std"),
    
    oil_temp_mean=("oil_temp_c", "mean"),
    oil_temp_max=("oil_temp_c", "max"),
    oil_temp_std=("oil_temp_c", "std")
).reset_index()

In [64]:
oil_agg.head()

,transformer_id,gas_mean,gas_max,gas_std,oil_temp_mean,oil_temp_max,oil_temp_std
0,TXR-0001,143.895994,147.817787,3.216604,63.097518,67.064311,4.360108
1,TXR-0002,100.991994,123.367978,15.073843,52.544203,65.398255,6.346153
2,TXR-0003,103.183072,122.118616,17.408723,57.813396,67.197059,5.172855
3,TXR-0004,125.595390,151.866470,19.034478,42.952912,49.972436,5.498816
4,TXR-0005,115.326868,202.066251,43.176527,51.196584,53.452318,1.964975


In [65]:
print("Shape:", oil_agg.shape)

Shape: (232, 7)


In [66]:
# Aggregate Load Logs

load_agg = load_clean.groupby("transformer_id").agg(
    load_mean=("load_pct_of_rated", "mean"),
    load_max=("load_pct_of_rated", "max"),
    load_std=("load_pct_of_rated", "std")
).reset_index()


In [67]:
load_agg.head()

,transformer_id,load_mean,load_max,load_std
0,TXR-0001,94.450842,104.927908,8.915228
1,TXR-0002,65.642144,82.275383,8.850842
2,TXR-0003,71.619793,82.604979,10.675000
3,TXR-0004,50.071346,70.107299,9.690446
4,TXR-0005,64.048775,73.754073,8.496023


In [68]:
print("Shape:", load_agg.shape)

Shape: (232, 4)


#### Merge the aggregated data

In [69]:
# merge registry with oil
features = registry_clean.merge(
    oil_agg,
    on="transformer_id",
    how="left"
)

In [70]:
# add load
features = features.merge(
    load_agg,
    on="transformer_id",
    how="left"
)

In [71]:
features.head()

,transformer_id,install_year,rated_kva,cooling_type,zone,gas_mean,gas_max,gas_std,oil_temp_mean,oil_temp_max,oil_temp_std,load_mean,load_max,load_std
0,TXR-0001,2012,250,ONAF,Zone 3,143.895994,147.817787,3.216604,63.097518,67.064311,4.360108,94.450842,104.927908,8.915228
1,TXR-0002,2019,500,ONAN,Zone 3,100.991994,123.367978,15.073843,52.544203,65.398255,6.346153,65.642144,82.275383,8.850842
2,TXR-0003,2009,500,OFAF,Zone 1,103.183072,122.118616,17.408723,57.813396,67.197059,5.172855,71.619793,82.604979,10.675000
3,TXR-0004,2008,100,ONAF,Zone 1,125.595390,151.866470,19.034478,42.952912,49.972436,5.498816,50.071346,70.107299,9.690446
4,TXR-0005,2006,250,ONAN,Zone 2,115.326868,202.066251,43.176527,51.196584,53.452318,1.964975,64.048775,73.754073,8.496023


In [72]:
print("Shape:", features.shape)

Shape: (232, 14)


In [73]:
# Check transformer ID uniqueness

print("Unique transformer IDs:", features["transformer_id"].nunique())
print("Total rows:", len(features))

Unique transformer IDs: 232
Total rows: 232


In [74]:
print("\nMissing values:")
print(features.isna().sum())


Missing values:
transformer_id    0
install_year      0
rated_kva         0
cooling_type      0
zone              0
gas_mean          0
gas_max           0
gas_std           0
oil_temp_mean     0
oil_temp_max      0
oil_temp_std      0
load_mean         0
load_max          0
load_std          0
dtype: int64


#### merging labels_clean onto features

In [75]:
labeled_data = features.merge(
    labels_clean,
    on="transformer_id",
    how="inner"
)

In [76]:
labeled_data.head()

,transformer_id,install_year,rated_kva,cooling_type,zone,gas_mean,gas_max,gas_std,oil_temp_mean,oil_temp_max,oil_temp_std,load_mean,load_max,load_std,fault_status
0,TXR-0001,2012,250,ONAF,Zone 3,143.895994,147.817787,3.216604,63.097518,67.064311,4.360108,94.450842,104.927908,8.915228,Fault
1,TXR-0002,2019,500,ONAN,Zone 3,100.991994,123.367978,15.073843,52.544203,65.398255,6.346153,65.642144,82.275383,8.850842,Fault
2,TXR-0003,2009,500,OFAF,Zone 1,103.183072,122.118616,17.408723,57.813396,67.197059,5.172855,71.619793,82.604979,10.675000,Fault
3,TXR-0004,2008,100,ONAF,Zone 1,125.595390,151.866470,19.034478,42.952912,49.972436,5.498816,50.071346,70.107299,9.690446,Normal
4,TXR-0005,2006,250,ONAN,Zone 2,115.326868,202.066251,43.176527,51.196584,53.452318,1.964975,64.048775,73.754073,8.496023,Normal


In [77]:
print("Shape:", labeled_data.shape)

Shape: (198, 15)


In [78]:
labeled_data["fault_status"].value_counts()

Normal    149
Fault      49
Name: fault_status, dtype: int64

#### Now Create the prediction dataset

In [79]:
prediction_data = features.merge(
    prediction_clean,
    on="transformer_id",
    how="inner"
)

In [80]:
prediction_data.head()

,transformer_id,install_year,rated_kva,cooling_type,zone,gas_mean,gas_max,gas_std,oil_temp_mean,oil_temp_max,oil_temp_std,load_mean,load_max,load_std
0,TXR-0012,2003,500,OFAF,Zone 1,47.350131,64.188311,17.744506,50.263817,63.002892,9.073555,66.213579,82.120859,7.552936
1,TXR-0022,2009,500,Unknown,Zone 2,126.921125,142.756378,16.361160,57.777658,64.305629,6.002222,75.484421,89.516598,9.696194
2,TXR-0049,2019,1000,OFAF,Zone 2,168.774258,188.443690,12.657774,48.918014,54.264071,3.138218,61.161177,68.575014,8.873620
3,TXR-0050,2010,100,OFAF,Zone 2,51.557031,67.917595,15.426892,51.863522,55.747815,3.713548,58.607323,78.549050,8.272227
4,TXR-0055,2011,500,OFAF,Zone 1,110.915097,117.383762,4.695938,53.435380,59.668077,4.308469,69.526657,76.936889,6.127914


In [81]:
print("Shape:", prediction_data.shape)

Shape: (34, 14)


In [82]:
print("Unique transformer IDs:",
      prediction_data["transformer_id"].nunique())

Unique transformer IDs: 34


In [83]:
print("Missing values:")
print(prediction_data.isna().sum())

print("\nTarget column present:",
      "fault_status" in prediction_data.columns)

Missing values:
transformer_id    0
install_year      0
rated_kva         0
cooling_type      0
zone              0
gas_mean          0
gas_max           0
gas_std           0
oil_temp_mean     0
oil_temp_max      0
oil_temp_std      0
load_mean         0
load_max          0
load_std          0
dtype: int64

Target column present: False


#### Save labeled & prediction_data inside processed_data folder in data folder


In [84]:
processed_path = Path("../data/processed_data")
processed_path.mkdir(parents=True, exist_ok=True)

In [85]:
labeled_data.to_csv(processed_path / "labeled_data.csv", index=False)
prediction_data.to_csv(processed_path / "prediction_data.csv", index=False)

In [86]:
print("Labeled data:", labeled_data.shape)
print("Prediction data:", prediction_data.shape)

Labeled data: (198, 15)
Prediction data: (34, 14)
